RQs:
- What is the distribution of contract type between occupations/sectors?
- What is the distribution of salary between occupations/sectors?
    - Avg annual increase by sector?
    - Stretch: hourly pay / annual pay / annualised pay?
- What proportion of ads in each occupation/sector mention learning and development/career progression/CPD?
- What proportion of ads in each occupation/sector mention flexible hours/flexible shifts?

Stretch: unsupervised approach: topic modelling of JQ sentences in early years sector vs a comparison sector

Limitations:
- The sample only covers the years 2021-2023 inclusive

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from typing import List

from dap_job_quality import PROJECT_DIR
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.utils import analysis_utils

In [ ]:
def plot_proportion(df, dim='FLEX_HOURS', save_path=None):

    filtered_df = df[df[dim] == 1.0]

    count_df = filtered_df.groupby('sector')['id'].count()

    total_count = df.groupby('sector')['id'].count()

    proportion_df = (count_df / total_count) * 100
    proportion_df = proportion_df.sort_values(ascending=False)

    _, ax = plt.subplots(figsize=(12, 8))

    bars = ax.barh(proportion_df.index, proportion_df.values, color='skyblue', edgecolor='black')

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.0f}%', 
                va='center', ha='left', color='black', fontsize=10)
    
    ax.set_xlabel('Proportion (%)')
    ax.set_ylabel('Sector')
    ax.set_title(f'Proportion of job ads mentioning {dim} by Sector')
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')

    plt.tight_layout()
    plt.show()

In [ ]:
# Produced by pipeline/eyp/stratified_sample.py
afs_raw_sample = pd.read_parquet('s3://open-jobs-lake/job_quality/early_years/evaluation_sample/job_ads_sample_size_16392.parquet')
# Produced by pipeline/find_job_quality.py
processed_ads = pd.read_parquet('s3://open-jobs-lake/job_quality/outputs/afs/job_ads_prod_True_n_16392.parquet')
# Produced by pipeline/eyp/eyp_auxiliary_data.py
salary_info = pd.read_parquet('s3://open-jobs-lake/job_quality/early_years/evaluation_sample/job_ads_sample_size_16392_metadata_production_True.parquet')
lookup = get_keywords()

In [ ]:
print(f"The complete sample includes {len(afs_raw_sample)} job adverts")

In [ ]:
len(processed_ads['id'].unique())

In [ ]:
print(f"The sectors to which EYP roles are compared are: {afs_raw_sample['sector'].unique()}")

Data for [this visualisation](https://app.flourish.studio/visualisation/19037076/edit):

In [ ]:
sample_sector_by_itl = afs_raw_sample.groupby(['sector', 'itl_1_name']).agg('size').unstack(fill_value=0)
sample_sector_by_itl.to_csv('outputs/sector_itl1.csv')
sample_sector_by_itl

Data for [this visualisation](https://app.flourish.studio/visualisation/19037320/edit):

In [ ]:
distribution_itl1 = afs_raw_sample.groupby(['itl_1_name']).agg('size')
distribution_itl1.to_csv('itl1.csv')
distribution_itl1

In [ ]:
processed_ads = processed_ads[['id', 'sentences_split', 'target_phrase']].drop_duplicates()

In [ ]:
processed_ads = pd.merge(processed_ads, lookup[['target_phrase', 'subcategory','dimension']], on='target_phrase', how='left')

In [ ]:
dimensions_wide = analysis_utils.create_wide_table(processed_ads)

In [ ]:
afs_sample = pd.merge(afs_raw_sample, dimensions_wide, on='id', how='left')

In [ ]:
columns_to_replace = dimensions_wide.columns[1:] # the first column is the id

In [ ]:
# These columns have NaN where there are *no* mentions of JQ dimensions in these job adverts
afs_sample[columns_to_replace] = afs_sample[columns_to_replace].fillna(0)

In [ ]:
plot_proportion(afs_sample, 'FLEX_HOURS', save_path='outputs/prop_flex_hours_by_sector.png')

In [ ]:
plot_proportion(afs_sample, 'L&D', save_path='outputs/prop_l&d_by_sector.png')

# Clean and analyse salary data

In [ ]:
# Check the distribution of different salary rates
salary_info['raw_salary_unit'].value_counts(dropna=False)

In [ ]:
missing_salary_df = salary_info[salary_info['raw_salary_unit'].isna()]

In [ ]:
analysis_utils.extract_salary_info("From £9.50 per hour")

In [ ]:
salary_descs = missing_salary_df['description'].tolist()
salaries = [analysis_utils.extract_salary_info(desc) for desc in salary_descs]
salaries

In [ ]:
# An example of an ad that includes a £3000 refer-a-friend bonus
missing_salary_df.iloc[8]['description']

In [ ]:
data = []
for sublist in salaries:
    if sublist:
        data.extend(sublist)
    else:
        data.append({'min_salary': None, 'max_salary': None, 'rate': None})

In [ ]:
extracted_salaries_df = pd.DataFrame(data)
extracted_salaries_df['salary'] = extracted_salaries_df['min_salary'].fillna(0)
extracted_salaries_df.head()

In [ ]:
missing_salary_df['raw_salary_unit'] = extracted_salaries_df['rate']
missing_salary_df['raw_salary_float'] = extracted_salaries_df['salary']
missing_salary_df['raw_min_salary_float'] = extracted_salaries_df['min_salary']
missing_salary_df['raw_max_salary_float'] = extracted_salaries_df['max_salary']

In [ ]:
enhanced_salary_data = pd.concat([missing_salary_df, salary_info[~salary_info['raw_salary_unit'].isna()]])
enhanced_salary_data['raw_salary_unit'] = enhanced_salary_data['raw_salary_unit'].str.lower()
enhanced_salary_data['raw_salary_unit'] = enhanced_salary_data['raw_salary_unit'].replace('annum', 'year')
enhanced_salary_data['raw_salary_unit'] = enhanced_salary_data['raw_salary_unit'].replace([np.nan, None, 'unknown'], np.nan)

In [ ]:
enhanced_salary_data['raw_salary_unit'].value_counts(dropna=False)

In [ ]:
# Check how many ads have the salary expressed per annum
enhanced_salary_data['is_annualised'] = enhanced_salary_data['raw_salary_unit'] == 'year'
enhanced_salary_data['is_annualised'].value_counts(dropna=False)

In [ ]:
afs_sample_enhanced = pd.merge(afs_raw_sample, enhanced_salary_data[['id', 'raw_salary_unit', 'raw_salary_float', 'raw_min_salary_float', 'raw_max_salary_float','is_annualised']], on='id', how='left')

In [ ]:
# Check that the merge was 1-1, and did not introduce duplicates
len(afs_sample_enhanced) - len(afs_raw_sample)

In [ ]:
# Filter the data to only records that have some kind of salary rate unit
afs_sample_enhanced_w_salaries = afs_sample_enhanced[afs_sample_enhanced['raw_salary_unit'].notna()]

In [ ]:
# proportion of ads offering hourly/daily/yearly salary
grouped = afs_sample_enhanced_w_salaries.groupby(['sector', 'raw_salary_unit']).size().unstack(fill_value=0)
grouped

In [ ]:
grouped.to_csv('outputs/salary_unit_by_sector.csv')

In [ ]:
afs_sample_enhanced_w_salaries['raw_salary_float'] = afs_sample_enhanced_w_salaries['raw_salary_float'].fillna(afs_sample_enhanced_w_salaries['raw_min_salary_float'])

afs_sample_enhanced_w_salaries['hourly_wage'] = afs_sample_enhanced_w_salaries.apply(analysis_utils.calculate_hourly_wage, axis=1)

In the next chunks we check the distribution of hourly wage and exclude outlying data.

In [ ]:
afs_sample_enhanced_w_salaries['hourly_wage'].describe()

In [ ]:
afs_sample_enhanced_w_salaries[afs_sample_enhanced_w_salaries['hourly_wage']<10]['hourly_wage'].hist(bins=100)

In [ ]:
afs_sample_enhanced_w_salaries[afs_sample_enhanced_w_salaries['hourly_wage']>20]['hourly_wage'].hist(bins=100)

In [ ]:
# Remove outlying salaries from the data.
# The minimum wage for 16 year olds in 2023 was £5.28 so realistically there shouldn't be hourly pay much lower than this.
afs_sample_enhanced_w_salaries = afs_sample_enhanced_w_salaries[(afs_sample_enhanced_w_salaries['hourly_wage']>=5)&(afs_sample_enhanced_w_salaries['hourly_wage']<50)]

In [ ]:
# Export data for the boxplot of hourly wage by sector
afs_sample_enhanced_w_salaries[['id', 'sector', 'hourly_wage']].to_csv('outputs/salary_boxplot.csv')

In [ ]:
# Data for the line plot of median salary by sector over time
afs_sample_enhanced_w_salaries.groupby(['year', 'sector']).agg({'hourly_wage': 'median'}).unstack(fill_value=0).to_csv('outputs/hourly_wage_by_year_and_sector.csv')

In [ ]:
afs_sample_enhanced_w_salaries_dimensions = pd.merge(afs_sample_enhanced_w_salaries, dimensions_wide, on='id', how='left')

In [ ]:
len(afs_sample_enhanced_w_salaries_dimensions) - len(afs_sample_enhanced_w_salaries)

In [ ]:
afs_sample_enhanced_w_salaries_dimensions['sector'].value_counts()

In [ ]:
hourly_wage_v_flex_hours = afs_sample_enhanced_w_salaries_dimensions.groupby('sector').agg({'hourly_wage': 'median', 'FLEX_HOURS': 'sum', 'sector': 'size'})
hourly_wage_v_flex_hours['prop_flex_hours'] = hourly_wage_v_flex_hours['FLEX_HOURS'] / hourly_wage_v_flex_hours['sector']
hourly_wage_v_flex_hours

hourly_wage_v_flex_hours.to_csv('outputs/hourly_wage_and_flex_hours_by_sector.csv')

In [ ]:
hourly_wage_v_ld = afs_sample_enhanced_w_salaries_dimensions.groupby('sector').agg({'hourly_wage': 'median', 'L&D': 'sum', 'sector': 'size'})
hourly_wage_v_ld['prop_l&d'] = hourly_wage_v_ld['L&D'] / hourly_wage_v_ld['sector']
hourly_wage_v_ld.to_csv('outputs/hourly_wage_and_l&d_by_sector.csv')

# Contract type

In [ ]:
contract_refs = processed_ads[processed_ads['subcategory']=='CONTRACT']

In [ ]:
contract_refs.columns

In [ ]:
# Check most frequent words and then we'll do a basic regex for these frequently occurring terms
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = " ".join(
                contract_refs[
                    "sentences_split"
                ].tolist()
                )

# Generate a word cloud image
wordcloud = WordCloud().generate(text)

# Display the generated image:

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")

In [ ]:
analysis_utils.classify_contract_type('Care Assistant  Guaranteed minimum 20-hour contact Salary  £11per hour weekday evenings and  £12 per hour weekends Hours 5 00 pm to 10 00 pm evenings and alternate weekends Location  Chichester, Bognor Regis and surrounding areas.')

In [ ]:
contract_refs['classified_contract_type'] = contract_refs['sentences_split'].apply(analysis_utils.classify_contract_type)
contract_refs['classified_contract_type'].value_counts()

In [ ]:
#Inspect the ones that couldn't be identified
contract_refs[contract_refs['classified_contract_type']=='Unknown']['sentences_split'].to_list()

In [ ]:
contract_refs['classified_contract_type'].value_counts()

In [ ]:

# Assuming your DataFrame is named df
# Step 1: Group by 'id' and 'classified_contract_type', then count occurrences
counts = contract_refs.groupby(['id', 'classified_contract_type']).size().reset_index(name='count')

# Step 2: Find the most frequent 'classified_contract_type' for each 'id'
most_frequent = counts.loc[counts.groupby('id')['count'].idxmax()]

# Step 3: Group the original DataFrame by 'id' and aggregate 'classified_contract_type' into a list
contract_df = contract_refs.groupby('id')['classified_contract_type'].agg(list).reset_index()

# Step 4: Define a function to determine the final classified contract type
def determine_contract_type(types: List[str]):
    if 'Temporary' in types:
        return 'Temporary'
    else:
        # Count occurrences of each type
        type_counts = pd.Series(types).value_counts()
        most_common = type_counts.idxmax()
        if len(type_counts) == 1:  # Only one unique type
            return most_common
        elif len(type_counts) > 1:
            # If the list contains more than one type, we need to check the most common
            if most_common == 'Permanent' or most_common == 'Unknown':
                return most_common
            else:
                return 'Unknown'  # Fallback to 'Unknown' if neither 'Temporary' nor most frequent matches
        return 'Unknown'

# Step 5: Apply the function to determine the final contract type for each group
contract_df['final_contract_type'] = contract_df['classified_contract_type'].apply(determine_contract_type)

In [ ]:
contract_df

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract = pd.merge(afs_sample_enhanced_w_salaries_dimensions, contract_df[['id', 'final_contract_type']], on='id', how='left')

In [ ]:
afs_sample_enhanced_w_salaries_dimensions_contract.groupby(['sector', 'final_contract_type']).size()

In [ ]:
contract_prop_df = afs_sample_enhanced_w_salaries_dimensions_contract.groupby(['sector', 'final_contract_type']).size().unstack(fill_value=0)
contract_prop_df

In [ ]:
contract_prop_df[['Permanent', 'Temporary']].to_csv('outputs/sector_contract_type.csv')

# Functions to extract hours

In [ ]:
def match_hours_per_week(text):
    
    working_hours = []
    
    hours_pattern = re.compile(r"(\d{1,2}(?:-\d{1,2})?)\s*(hours|hrs|h|hour)\s*(per\s*week|p\/w|a\s*week)?", re.IGNORECASE)
    
    explicit_hours = hours_pattern.findall(text)
    
    if explicit_hours:
        for match in explicit_hours:
            hours_range = match[0]
            if '-' in hours_range:
                min_hours, max_hours = map(int, hours_range.split('-'))
            else:
                min_hours = max_hours = int(hours_range)
            working_hours.append({'min_hours': min_hours, 'max_hours': max_hours})
            
    return working_hours

# hours_texts = ['40 hours per week',
# 'Full time Hours - 8:30am - 4:30pm',
# 'Working days : Tuesday 10am-2pm',
# '35 hours per week',
# 'Full-time contract of employment, 39 hours per week',
# '40-45 hours p/w',
# 'Full-time',
# '4 Days a Week',
# '4 days per week Monday – Thursday, 8.30AM – 3.30PM',
# 'part-time 24 hour per week contract',
# 'Working hours: 40 hours over 5 out of 7 days']

# [match_hours_per_week(text) for text in hours_texts]

def parse_time(time_str):
    # Define time formats to match various input patterns
    time_formats = [
        "%I:%M%p",  # e.g., "8:30am"
        "%I.%M%p",  # e.g., "8.30am"
        "%I:%M %p",  # e.g., "8:30 am"
        "%I.%M %p",  # e.g., "8.30 am"
        "%I%p",      # e.g., "8am"
        "%I %p",     # e.g., "8 am"
        "%I:%M",     # e.g., "8:30"
        "%I.%M",     # e.g., "8.30"
        "%I"         # e.g., "8"
    ]
    
    for time_format in time_formats:
        try:
            return datetime.strptime(time_str, time_format)
        except ValueError:
            continue
    return None

def extract_time_difference(text):
    # Regex pattern to capture time ranges
    time_range_pattern = re.compile(
        r"(\d{1,2}[:\.]\d{2}|\d{1,2})(?:\s*[apAP][mM])?\s*-\s*(\d{1,2}[:\.]\d{2}|\d{1,2})(?:\s*[apAP][mM])?",
        re.IGNORECASE
    )

    matches = time_range_pattern.findall(text)
    if not matches:
        return None

    for start_time_str, end_time_str in matches:
        # Clean up and parse the times
        start_time = parse_time(start_time_str.strip())
        end_time = parse_time(end_time_str.strip())

        if not start_time or not end_time:
            continue

        # If end time is earlier than start time, assume it spans to the next day (24-hour format)
        if end_time <= start_time:
            end_time += timedelta(hours=12)

        # Calculate the time difference in hours
        time_difference = (end_time - start_time).seconds / 3600.0
        return time_difference

    return None

texts = [
    "8.30-11.30am",
    "8.30am - 12.30pm",
    "8.30 am - 12.30 pm",
    "8:30 am - 12:30 pm",
    "12-6",
    "8 am - 12 pm"
]

for text in texts:
    time_diff = extract_time_difference(text)
    print(f"'{text}' => {time_diff} hours")
    
def count_working_days(text):
    # Define day mappings to handle various abbreviations
    day_mappings = {
        'monday': 0, 'mon': 0, 'mondays': 0,
        'tuesday': 1, 'tue': 1, 'tues': 1, 'tuesdays': 1,
        'wednesday': 2, 'wed': 2, 'wednesdays': 2,
        'thursday': 3, 'thu': 3, 'thurs': 3, 'thursdays': 3,
        'friday': 4, 'fri': 4, 'fridays': 4,
        'saturday': 5, 'sat': 5, 'saturdays': 5,
        'sunday': 6, 'sun': 6, 'sundays': 6
    }
    
    # Handle ranges like "Mon - Thurs"
    range_pattern = re.compile(r"(\b\w+\b)\s*(-|to)\s*(\b\w+\b)", re.IGNORECASE)
    # Handle individual days or day lists like "Mondays, Tuesdays and Fridays"
    individual_days_pattern = re.compile(r"\b\w+\b", re.IGNORECASE)

    days = set()

    # Check for ranges
    if len(range_pattern.findall(text))>0:
        # print(range_pattern.findall(text))
        for match in range_pattern.findall(text):
            start_day = day_mappings.get(match[0].lower())
            end_day = day_mappings.get(match[2].lower())

            if start_day is not None and end_day is not None:
                # Add all days in the range to the set
                if start_day <= end_day:
                    for day in range(start_day, end_day + 1):
                        days.add(day)
                else:
                    # Handles cases where the range might go from end of the week to the start (e.g., Fri - Mon)
                    for day in range(start_day, 7):
                        days.add(day)
                    for day in range(0, end_day + 1):
                        days.add(day)
    else:
        # Check for individual days
        for day in individual_days_pattern.findall(text):
            day_index = day_mappings.get(day.lower())
            if day_index is not None:
                days.add(day_index)
    
    # print(days)
    return len(days)

# # Example usage
# texts = [
#     "Monday - Thursday",
#     "Mon - Thurs",
#     "Mondays, Tuesdays and Fridays",
#     "Fri - Mon",
#     "Tuesday to Friday",
#     "Saturday - Sunday",
#     "Wed, Thu, Fri"
# ]

# for text in texts:
#     day_count = count_working_days(text)
#     print(f"'{text}' => {day_count} days")

def check_full_time(text):
    return 'full time' in text.lower() or 'full-time' in text.lower()


In [ ]:
hours_refs = processed_ads[processed_ads['subcategory']=='HOURS']
hours_refs

In [ ]:
hours_refs['hours_per_week'] = hours_refs['sentences_split'].apply(match_hours_per_week)
hours_refs['working_days'] = hours_refs['sentences_split'].apply(count_working_days)
hours_refs['is_full_time'] = hours_refs['sentences_split'].apply(check_full_time)
hours_refs

In [ ]:
# Function to calculate 'hr_per_week_final'
def calculate_hr_per_week_final(row):
    # Rule 1: Use 'hours_per_week' if available
    if row['hours_per_week']:
        # Assuming 'hours_per_week' contains a list of dictionaries with 'min_hours' and 'max_hours'
        return row['hours_per_week'][0]['min_hours']  # or average, max, etc., depending on your requirement
    # Rule 2: Use 37.5 hours if 'is_full_time' is True
    elif row['is_full_time']:
        return 37.5  # or another standard full-time hours value
    # Rule 3: Use 'working_days' multiplied by 7.5
    elif isinstance(row['working_days'], int) and row['working_days'] > 0:
        return row['working_days'] * 7.5
    # If none of the above rules apply, return None or 0
    else:
        return None

# Apply the function to each row of the DataFrame
hours_refs['hr_per_week_final'] = hours_refs.apply(calculate_hr_per_week_final, axis=1)

# Display the updated DataFrame
hours_refs


In [ ]:
hours_refs['hr_per_week_final'].value_counts(dropna=False)